In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


plt.rc('figure', figsize=(10, 6))

from matplotlib import rcParams
rcParams['font.family'] = 'New Gulim'
rcParams['font.size'] = 10
rcParams['axes.unicode_minus'] = False

In [2]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# log 값 변환 시 NaN등의 이슈로 log() 가 아닌 log1p() 를 이용하여 RMSLE 계산
def rmsle(y, pred):
    log_y = np.log1p(y)
    log_pred = np.log1p(pred)
    squared_error = (log_y - log_pred) ** 2
    rmsle = np.sqrt(np.mean(squared_error))
    return rmsle

# 사이킷런의 mean_squared_error() 를 이용하여 RMSE 계산
def rmse(y, pred):
    return np.sqrt(mean_squared_error(y, pred))

# RMSLE, RMSE, MAE 를 모두 계산
def evaluate_regr(y, pred):
    rmsle_val = rmsle(y, pred)
    rmse_val  = rmse(y, pred)
    mae_val   = mean_absolute_error(y, pred)
    print('RMSLE: {:.3f}, RMSE: {:.3f}, MAE: {:.3f}'.format(rmsle_val, rmse_val, mae_val))

In [3]:
# 데이터 로딩
bike_df = pd.read_csv('data/bike_train.csv')
print(bike_df.shape)
bike_df.head()

(10886, 12)


,datetime,season,holiday,workingday,weather,temp,atemp,humidity,windspeed,casual,registered,count
0,2011-01-01 00:00:00,1,0,0,1,9.84,14.395,81,0.0,3,13,16
1,2011-01-01 01:00:00,1,0,0,1,9.02,13.635,80,0.0,8,32,40
2,2011-01-01 02:00:00,1,0,0,1,9.02,13.635,80,0.0,5,27,32
3,2011-01-01 03:00:00,1,0,0,1,9.84,14.395,75,0.0,3,10,13
4,2011-01-01 04:00:00,1,0,0,1,9.84,14.395,75,0.0,0,1,1


In [4]:
import numpy as np
import pandas as pd

In [5]:
print(bike_df.columns.tolist())
print("shape:", bike_df.shape)
print(bike_df.dtypes)

['datetime', 'season', 'holiday', 'workingday', 'weather', 'temp', 'atemp', 'humidity', 'windspeed', 'casual', 'registered', 'count']
shape: (10886, 12)
datetime          str
season          int64
holiday         int64
workingday      int64
weather         int64
temp          float64
atemp         float64
humidity        int64
windspeed     float64
casual          int64
registered      int64
count           int64
dtype: object


In [6]:

# [STEP 2] 누수(leakage) 확인 + 결측치 점검
# 1) casual + registered = count 관계 확인 (대부분 동일하면 누수 확정)
leak_equal_cnt = ((bike_df["casual"] + bike_df["registered"]) == bike_df["count"]).sum()
total = len(bike_df)
print(f"leak check: (casual+registered)==count  -> {leak_equal_cnt}/{total} ({leak_equal_cnt/total:.4%})")

# 2) 결측치 확인
print("\n[missing values]")
print(bike_df.isnull().sum().sort_values(ascending=False).head(20))

leak check: (casual+registered)==count  -> 10886/10886 (100.0000%)

[missing values]
datetime      0
season        0
holiday       0
workingday    0
weather       0
temp          0
atemp         0
humidity      0
windspeed     0
casual        0
registered    0
count         0
dtype: int64


In [7]:
bike_step3 = bike_df.copy()

# 1) datetime 문자열 -> datetime 타입 변환
bike_step3["datetime"] = pd.to_datetime(bike_step3["datetime"])
bike_step3.info()

<class 'pandas.DataFrame'>
RangeIndex: 10886 entries, 0 to 10885
Data columns (total 12 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   datetime    10886 non-null  datetime64[us]
 1   season      10886 non-null  int64         
 2   holiday     10886 non-null  int64         
 3   workingday  10886 non-null  int64         
 4   weather     10886 non-null  int64         
 5   temp        10886 non-null  float64       
 6   atemp       10886 non-null  float64       
 7   humidity    10886 non-null  int64         
 8   windspeed   10886 non-null  float64       
 9   casual      10886 non-null  int64         
 10  registered  10886 non-null  int64         
 11  count       10886 non-null  int64         
dtypes: datetime64[us](1), float64(3), int64(8)
memory usage: 1020.7 KB


In [8]:

# 2) 파생 피처 생성
bike_step3["year"] = bike_step3["datetime"].dt.year
bike_step3["month"] = bike_step3["datetime"].dt.month
bike_step3["day"] = bike_step3["datetime"].dt.day
bike_step3["hour"] = bike_step3["datetime"].dt.hour
bike_step3["dayofweek"] = bike_step3["datetime"].dt.dayofweek  # 월=0, 요일

In [9]:
bike_step3['workingday'].value_counts()

workingday
1    7412
0    3474
Name: count, dtype: int64

In [10]:
bike_step3['holiday'].value_counts()

holiday
0    10575
1      311
Name: count, dtype: int64

In [11]:
bike_step3.head()

,datetime,season,holiday,workingday,weather,temp,atemp,humidity,windspeed,casual,registered,count,year,month,day,hour,dayofweek
0,2011-01-01 00:00:00,1,0,0,1,9.84,14.395,81,0.0,3,13,16,2011,1,1,0,5
1,2011-01-01 01:00:00,1,0,0,1,9.02,13.635,80,0.0,8,32,40,2011,1,1,1,5
2,2011-01-01 02:00:00,1,0,0,1,9.02,13.635,80,0.0,5,27,32,2011,1,1,2,5
3,2011-01-01 03:00:00,1,0,0,1,9.84,14.395,75,0.0,3,10,13,2011,1,1,3,5
4,2011-01-01 04:00:00,1,0,0,1,9.84,14.395,75,0.0,0,1,1,2011,1,1,4,5


In [14]:
from sklearn.model_selection import train_test_split

bike_step4 = bike_step3.copy()

# 1) 누수 컬럼 및 원본 datetime 제거
drop_cols = ["datetime", "casual", "registered"]
X = bike_step4.drop(columns=["count"] + drop_cols)
y = bike_step4["count"].copy()

# 2) 홀드아웃 분리
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("X_train shape:", X_train.shape)
print("X_valid shape:", X_valid.shape)
print("y_train shape:", y_train.shape)
print("y_valid shape:", y_valid.shape)

print("\n[train columns]")
print(X_train.columns.tolist())

X_train shape: (8708, 13)
X_valid shape: (2178, 13)
y_train shape: (8708,)
y_valid shape: (2178,)

[train columns]
['season', 'holiday', 'workingday', 'weather', 'temp', 'atemp', 'humidity', 'windspeed', 'year', 'month', 'day', 'hour', 'dayofweek']


In [15]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

# 1) 컬럼 타입 구분
categorical_cols = ["season","holiday","workingday","weather","year","month","day","hour","dayofweek"]
numeric_cols = ["temp","atemp","humidity","windspeed"]

# 2) 전처리: 범주형 원핫 + 수치형 스케일
preprocess = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
        ("num", StandardScaler(), numeric_cols),
    ],
    remainder="drop"
)